In [1]:
#instlling required things
!apt-get install -y -qq nmap
!pip install streamlit pandas plotly requests -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package libpcap0.8:amd64.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../0-libpcap0.8_1.10.1-4ubuntu1.22.04.1_amd64.deb ...
Unpacking libpcap0.8:amd64 (1.10.1-4ubuntu1.22.04.1) ...
Selecting previously unselected package liblinear4:amd64.
Preparing to unpack .../1-liblinear4_2.3.0+dfsg-5_amd64.deb ...
Unpacking liblinear4:amd64 (2.3.0+dfsg-5) ...
Selecting previously unselected package liblua5.3-0:amd64.
Preparing to unpack .../2-liblua5.3-0_5.3.6-1build1_amd64.deb ...
Unpacking liblua5.3-0:amd64 (5.3.6-1build1) ...
Selecting previously unselected package lua-lpeg:amd64.
Preparing to unpack .../3-lua-lpeg_1.0.2-1_amd64.deb ...
Unpacking lua-lpeg:amd64 (1.0.2-1) ...
Selecting previously unselected package nmap-common.
Preparing to unpack .../4-nmap-common_7.91+dfsg1+really7.80+dfsg1-2ubuntu0.1_all.deb ...
Unpacking nmap-common (7.91+dfsg1+really7.80+dfsg1-2ubuntu0.1) ...
Selecting previously unselected pac

In [3]:
# importing libraries
import os
import subprocess
import streamlit
import pandas
import plotly
import requests

In [62]:
# loading secrets
from google.colab import userdata
import os

os.environ["VT_API_KEY"] = userdata.get("VT_API_KEY")
os.environ["GMAIL_SENDER"] = userdata.get("GMAIL_SENDER")
os.environ["GMAIL_PASSWORD"] = userdata.get("GMAIL_PASSWORD")
os.environ["GMAIL_RECIPIENT"] = userdata.get("GMAIL_RECIPIENT")
os.environ["SCAN_TARGETS"] = userdata.get("SCAN_TARGETS") # used scanme.nmap.org  for target

print("Secrets loaded from Colab")

Secrets loaded from Colab


In [60]:
# dashboard
%%writefile dashboard.py
import streamlit as st
import subprocess
import xml.etree.ElementTree as ET
import pandas as pd
import plotly.express as px
import requests
import os
import smtplib
from email.mime.text import MIMEText
from datetime import datetime

st.set_page_config(page_title="RiskAnalyser", layout="wide")

st.title("RiskAnalyser")
st.write("Web Vulnerability Scanner and Risk Dashboard")

# LOAD ENV
targets = os.environ.get("SCAN_TARGETS","").split(",")

VT_KEY = os.environ.get("VT_API_KEY","")

sender = os.environ.get("GMAIL_SENDER","")
password = os.environ.get("GMAIL_PASSWORD","")
receiver = os.environ.get("GMAIL_RECIPIENT","")

scan_folder = "scan_results"
os.makedirs(scan_folder, exist_ok=True)

# RISK SERVICES
risk_ports = {
    "ftp":3,
    "telnet":4,
    "ssh":1,
    "smtp":2,
    "rdp":4
}

# SESSION STATE
if "df" not in st.session_state:
    st.session_state.df = None

if "scan_time" not in st.session_state:
    st.session_state.scan_time = None


# SIDEBAR
st.sidebar.title("RiskAnalyser")

st.sidebar.subheader("⚙️ Status")

if VT_KEY:
    st.sidebar.success("API key ready")
else:
    st.sidebar.error("API key missing")

if sender and password and receiver:
    st.sidebar.success("Email ready")
else:
    st.sidebar.warning("Email not configured")

st.sidebar.write("Targets:", ", ".join(targets))

st.sidebar.subheader("🎯 Scan Controls")

run_button = st.sidebar.button("Run Scan")

st.sidebar.subheader("🔍 Filter Results")


# FUNCTIONS

def run_scan(target):

    file = scan_folder + "/" + target + ".xml"

    subprocess.run(
        ["nmap","-Pn","-sV","-oX",file,target],
        capture_output=True
    )

    return file


def parse_xml(file):

    rows = []

    root = ET.parse(file).getroot()

    for host in root.findall("host"):

        ip = host.find("address").get("addr")

        ports = host.findall(".//port")

        # if no open ports still show host
        if len(ports) == 0:

            rows.append({
                "ip": ip,
                "port": "None",
                "service": "No open ports"
            })

        for port in ports:

            svc = port.find("service")

            name = "unknown"

            if svc is not None:
                name = svc.get("name")

            rows.append({
                "ip": ip,
                "port": port.get("portid"),
                "service": name
            })

    return rows


def vt_check(ip):

    try:

        r = requests.get(
            "https://www.virustotal.com/api/v3/ip_addresses/" + ip,
            headers={"x-apikey":VT_KEY}
        )

        return r.json()["data"]["attributes"]["last_analysis_stats"]["malicious"]

    except:

        return 0


def risk(row):

    score = 1

    if row["service"] in risk_ports:
        score = score + risk_ports[row["service"]]

    score = score + row["malicious"]

    return score


def severity(score):

    if score <= 2:
        return "Low"

    if score <= 5:
        return "Medium"

    return "High"


def send_email(high_df):

    if len(high_df) == 0:
        return

    body = "High risk vulnerabilities detected\n\n"

    for i,row in high_df.iterrows():

        body += "IP: " + row["ip"] + "\n"
        body += "Port: " + str(row["port"]) + "\n"
        body += "Service: " + row["service"] + "\n"
        body += "Risk Score: " + str(row["risk"]) + "\n\n"

    msg = MIMEText(body)

    msg["Subject"] = "RiskAnalyser Alert"
    msg["From"] = sender
    msg["To"] = receiver

    try:

        server = smtplib.SMTP("smtp.gmail.com",587)

        server.starttls()

        server.login(sender,password)

        server.sendmail(sender,receiver,msg.as_string())

        server.quit()

    except:

        pass


# SAMPLE DATA BEFORE SCAN
if st.session_state.df is None:

    df = pd.DataFrame({
        "ip":["192.168.1.1","192.168.1.2","192.168.1.3"],
        "port":["22","80","21"],
        "service":["ssh","http","ftp"],
        "malicious":[0,1,3]
    })

    df["risk"] = df.apply(risk, axis=1)
    df["severity"] = df["risk"].apply(severity)

else:

    df = st.session_state.df


# RUN SCAN
if run_button:

    st.write("Scanning targets")

    rows = []

    for t in targets:

        st.write("Scanning " + t)

        xml = run_scan(t)

        rows.extend(parse_xml(xml))

    df = pd.DataFrame(rows)

    vt = {}

    for ip in df["ip"].unique():

        vt[ip] = vt_check(ip)

    df["malicious"] = df["ip"].map(vt)

    df["risk"] = df.apply(risk, axis=1)

    df["severity"] = df["risk"].apply(severity)

    st.session_state.df = df
    st.session_state.scan_time = datetime.now()

    send_email(df[df["severity"]=="High"])


# SIDEBAR FILTERS
ip_filter = st.sidebar.selectbox(
    "Filter by IP",
    ["All"] + list(df["ip"].unique())
)

service_filter = st.sidebar.selectbox(
    "Filter by Service",
    ["All"] + list(df["service"].unique())
)

severity_filter = st.sidebar.multiselect(
    "Filter by Severity",
    ["Low","Medium","High"],
    default=["Low","Medium","High"]
)

min_risk = st.sidebar.slider(
    "Min Risk Score",
    int(df["risk"].min()),
    int(df["risk"].max()),
    int(df["risk"].min())
)

filtered = df.copy()

if ip_filter != "All":
    filtered = filtered[filtered["ip"] == ip_filter]

if service_filter != "All":
    filtered = filtered[filtered["service"] == service_filter]

filtered = filtered[filtered["severity"].isin(severity_filter)]

filtered = filtered[filtered["risk"] >= min_risk]


# SHOW LAST SCAN
if st.session_state.scan_time:
    st.info("Last Scan: " + st.session_state.scan_time.strftime("%Y-%m-%d %H:%M"))


# KPI METRICS
st.subheader("Key Metrics")

c1,c2,c3 = st.columns(3)

c1.metric("Hosts", df["ip"].nunique())

c2.metric("Open Ports", len(df))

c3.metric("High Risk", len(df[df["severity"]=="High"]))


# TABS
tab1,tab2,tab3,tab4 = st.tabs([
    "Scan Results",
    "Charts",
    "Threat Analysis",
    "Export"
])


# TAB 1
with tab1:

    st.subheader("Scan Data")

    st.dataframe(filtered)

    st.subheader("Host Summary Table")

    summary = df.groupby("ip").agg(
        open_ports=("port","count"),
        max_risk=("risk","max"),
        services=("service",lambda x:", ".join(sorted(x.unique())))
    ).reset_index()

    st.dataframe(summary)


# TAB 2
with tab2:

    st.subheader("Charts")

    # 1 Open Ports per Host (Bar Chart)
    ports = df.groupby("ip")["port"].count().reset_index()

    fig1 = px.bar(
        ports,
        x="ip",
        y="port",
        title="Open Ports per Host",
        color="port"
    )

    st.plotly_chart(fig1, use_container_width=True)


    # 2 Services Distribution (Bar Chart)
    svc = df["service"].value_counts().reset_index()

    svc.columns = ["service","count"]

    fig2 = px.bar(
        svc,
        x="service",
        y="count",
        title="Services Found",
        color="count"
    )

    st.plotly_chart(fig2, use_container_width=True)


    # 3 Severity Distribution (Pie Chart)
    sev = df["severity"].value_counts().reset_index()

    sev.columns = ["severity","count"]

    fig3 = px.pie(
        sev,
        names="severity",
        values="count",
        title="Severity Distribution"
    )

    st.plotly_chart(fig3, use_container_width=True)


    # 4 Risk Score per Host (Bar Chart)
    risk_host = df.groupby("ip")["risk"].sum().reset_index()

    fig4 = px.bar(
        risk_host,
        x="ip",
        y="risk",
        title="Total Risk Score per Host",
        color="risk"
    )

    st.plotly_chart(fig4, use_container_width=True)


    # 5 Service vs Risk (Scatter Plot)
    fig5 = px.scatter(
        df,
        x="service",
        y="risk",
        color="severity",
        size="risk",
        title="Service vs Risk Score"
    )

    st.plotly_chart(fig5, use_container_width=True)


    # 6 Malicious Reports per Host
    vt = df.groupby("ip")["malicious"].max().reset_index()

    fig6 = px.bar(
        vt,
        x="ip",
        y="malicious",
        title="VirusTotal Malicious Reports"
    )

    st.plotly_chart(fig6, use_container_width=True)


    # 7 Risk vs Open Ports
    exposure = df.groupby("ip").agg(
        open_ports=("port","count"),
        total_risk=("risk","sum")
    ).reset_index()

    fig7 = px.scatter(
        exposure,
        x="open_ports",
        y="total_risk",
        text="ip",
        size="total_risk",
        title="Risk vs Exposure"
    )

    st.plotly_chart(fig7, use_container_width=True)


    # 8 Services Pie Chart
    fig8 = px.pie(
        svc,
        names="service",
        values="count",
        title="Service Share"
    )

    st.plotly_chart(fig8, use_container_width=True)


    # 9 Severity Histogram
    fig9 = px.histogram(
        df,
        x="risk",
        color="severity",
        title="Risk Score Distribution"
    )

    st.plotly_chart(fig9, use_container_width=True)

    vt_chart = df.groupby("ip").agg(
        malicious=("malicious","max"),
        risk=("risk","sum")
    ).reset_index()

    vt_chart.columns = ["IP","Malicious Reports","Total Risk"]

    fig10 = px.bar(
        vt_chart,
        x="IP",
        y=["Malicious Reports","Total Risk"],
        barmode="group",
        title="VirusTotal Detection vs Risk Score per Host"
    )

    st.plotly_chart(fig10, use_container_width=True)

# TAB 3
with tab3:

    st.subheader("Threat Analysis")

    high = df[df["severity"]=="High"]

    medium = df[df["severity"]=="Medium"]

    st.write("Top Risky Hosts")

    top = df.groupby("ip")["risk"].sum().reset_index()

    top = top.sort_values("risk",ascending=False).head(5)

    st.dataframe(top)

    st.write("High Risk Findings")

    st.dataframe(high)

    st.write("Medium Risk Findings")

    st.dataframe(medium)


# TAB 4
with tab4:

    st.subheader("Export")

    st.download_button(
        "Download Full CSV",
        df.to_csv(index=False),
        "scan_results.csv"
    )

    st.download_button(
        "Download Filtered CSV",
        filtered.to_csv(index=False),
        "filtered_results.csv"
    )

Overwriting dashboard.py


In [29]:
import os

os.makedirs(".streamlit", exist_ok=True)
config = """
[theme]
primaryColor="#4f8ef7"
backgroundColor="#0f1117"
secondaryBackgroundColor="#1a1d2e"
textColor="#e2e8f0"
"""
open(".streamlit/config.toml","w").write(config)

print("Theme ready")

Theme ready


In [42]:
import subprocess, threading, time

def run_streamlit():
    subprocess.run(['streamlit', 'run', 'dashboard.py',
                    '--server.port', '8501', '--server.headless', 'true'])

threading.Thread(target=run_streamlit, daemon=True).start()
time.sleep(5)
print('Streamlit running')
print('Now run Cell below to open the Cloudflare tunnel.')

Streamlit running
Now run Cell below to open the Cloudflare tunnel.


In [63]:
!streamlit run dashboard.py --server.port 8501 --server.headless true &>/content/log.txt &
!cloudflared tunnel --url http://localhost:8501

2026-03-14T15:52:36Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-14T15:52:36Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-14T15:52:40Z INF +--------------------------------------------------------------------------------------------+
2026-03-14T15:52:40Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-14T15:52:40Z INF |  https://knew-beautifully-wyoming-number.trycloudflare